# Exploration OpenAgenda

---

- **Projet 9 :** Concevez et déployez un système RAG
- **Auteur :** Justine Tranchant
- **Date :** mai 2026

---

Objectif : comprendre comment récupérer les événements publics OpenAgenda, tester les filtres utiles et identifier les champs à conserver pour la suite du projet RAG.

Ce notebook correspond à l'étape d'exploration. Il ne construit pas encore le client final ni l'index FAISS.

## Imports et configuration

On utilise `requests` pour appeler l'API, `pandas` pour inspecter les résultats, et `json` pour sauvegarder un échantillon brut.

In [1]:
import json
import sys

import pandas as pd
import requests

# Permet d'importer src.config quand le notebook est exécuté depuis le dossier notebooks/.
sys.path.append("..")

from src.config import (
    OPENAGENDA_BASE_URL,
    OPENAGENDA_CITIES,
    OPENAGENDA_ORDER_BY,
    OPENAGENDA_PAGE_SIZE,
    OPENAGENDA_USEFUL_FIELDS,
    PATHS,
    RAW_OPENAGENDA_SAMPLE_FILENAME,
)

from src.openagenda import build_openagenda_where_clause, save_openagenda_sample

In [2]:
PATHS.data_raw.mkdir(parents=True, exist_ok=True)

OPENAGENDA_BASE_URL

'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records'

## Villes ciblées

Pour le POC, on limite volontairement la zone géographique au Bassin d'Arcachon et à quelques villes proches.

Ce choix permet d'obtenir un jeu de données local, cohérent avec un assistant de recommandation d'événements.

In [3]:
OPENAGENDA_CITIES

['Arcachon',
 'La Teste-de-Buch',
 'Pyla-sur-Mer',
 'Gujan-Mestras',
 'Le Teich',
 'Biganos',
 'Audenge',
 'Lanton',
 'Andernos-les-Bains',
 'Arès',
 'Lège-Cap-Ferret',
 'Mios',
 'Marcheprime',
 'Salles',
 'Belin-Béliet',
 'Le Barp',
 'Lugos',
 'Saint-Magne']

## Construction du filtre `where`

Le filtre `refine` est pratique pour explorer des facettes, mais il n'est pas adapté pour exprimer proprement une condition de plage de dates.

Ici, on utilise donc `where` pour combiner :

- le département ;
- la liste des villes ;
- la contrainte temporelle : événements dont la première date est dans les 12 derniers mois.

In [4]:
where_clause = build_openagenda_where_clause()

print(where_clause)

location_department = "Gironde"
AND location_region = "Nouvelle-Aquitaine"
AND location_countrycode IN ("FR", "fr")
AND location_city IN ("Arcachon", "La Teste-de-Buch", "Pyla-sur-Mer", "Gujan-Mestras", "Le Teich", "Biganos", "Audenge", "Lanton", "Andernos-les-Bains", "Arès", "Lège-Cap-Ferret", "Mios", "Marcheprime", "Salles", "Belin-Béliet", "Le Barp", "Lugos", "Saint-Magne")
AND firstdate_begin >= now(years=-1)


## Premier appel API

On récupère un petit nombre d'événements pour vérifier que l'endpoint, les paramètres et le filtre fonctionnent.

In [5]:
params = {
    "limit": 20,
    "where": where_clause,
    "order_by": OPENAGENDA_ORDER_BY,
}

response = requests.get(OPENAGENDA_BASE_URL, params=params, timeout=30)
response.raise_for_status()

data = response.json()

data.keys()

dict_keys(['total_count', 'results'])

In [6]:
total_count = data.get("total_count", 0)
events = data.get("results", [])

print(f"Nombre total d'événements trouvés : {total_count}")
print(f"Nombre d'événements récupérés dans cette page : {len(events)}")

Nombre total d'événements trouvés : 603
Nombre d'événements récupérés dans cette page : 20


## Inspection rapide d'un événement

Cette cellule permet de visualiser la structure JSON brute renvoyée par l'API.

In [7]:
if events:
    print(json.dumps(events[0], ensure_ascii=False, indent=2))
else:
    print("Aucun événement trouvé avec ces filtres.")

{
  "uid": "78749329",
  "slug": "ballade-velo",
  "canonicalurl": "https://openagenda.com/maiavelo/events/ballade-velo",
  "title_fr": "BALLADE VELO -Ô PIGNON",
  "description_fr": "GARE DE CASSY/  ARES  BALADE/ REJOINDRE LA FETE DU VELO",
  "longdescription_fr": "<p>rejoindre fete du velo o pignon ares</p>",
  "conditions_fr": null,
  "keywords_fr": null,
  "image": "https://cdn.openagenda.com/main/dd4a0c77fc2541a2b8f9a4f08fb0ceb8.base.image.jpg",
  "imagecredits": null,
  "thumbnail": "https://cdn.openagenda.com/main/dd4a0c77fc2541a2b8f9a4f08fb0ceb8.thumb.image.jpg",
  "originalimage": "https://cdn.openagenda.com/main/dd4a0c77fc2541a2b8f9a4f08fb0ceb8.full.image.jpg",
  "updatedat": "2025-04-21T08:39:15+00:00",
  "daterange_fr": "Samedi 10 mai 2025, 06h30",
  "firstdate_begin": "2025-05-10T04:30:00+00:00",
  "firstdate_end": "2025-05-10T15:00:00+00:00",
  "lastdate_begin": "2025-05-10T04:30:00+00:00",
  "lastdate_end": "2025-05-10T15:00:00+00:00",
  "timings": "[{\"begin\": \"2025-05

## Conversion en DataFrame

Le DataFrame facilite l'inspection des colonnes et des valeurs manquantes.

In [8]:
df = pd.DataFrame(events)

df.shape

(20, 56)

In [9]:
df.head()

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,image,imagecredits,...,originagenda_uid,contributor_email,contributor_contactnumber,contributor_contactname,contributor_contactposition,contributor_organization,category,country_fr,registration,links
0,78749329,ballade-velo,https://openagenda.com/maiavelo/events/ballade...,BALLADE VELO -Ô PIGNON,GARE DE CASSY/ ARES BALADE/ REJOINDRE LA FET...,<p>rejoindre fete du velo o pignon ares</p>,NaN,None,https://cdn.openagenda.com/main/dd4a0c77fc2541...,NaN,...,6013618,None,None,None,None,None,None,France (Métropole),NaN,NaN
1,1832998,lichens-tresors-meconnus,https://openagenda.com/cbnsa/events/lichens-tr...,"Lichens, trésors méconnus",Balade botanique de 2h30 à la découverte des l...,<p>Saviez-vous que certains lichens sont utili...,"Gratuit, inscription obligatoire",None,https://cdn.openagenda.com/main/85d15b8921c24d...,CBNSA,...,46716941,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://www.girond...","[{""link"": ""https://www.gironde.fr/idees-de-sor..."
2,15770224,mai-a-velo-sud-bassin-darcachon,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à vélo - Sud Bassin d'Arcachon,Insercycles vous invite à découvrir ou redécou...,<p>Dans le cadre de l'événement national Mai à...,NaN,None,NaN,NaN,...,6013618,None,None,None,None,None,None,France (Métropole),"[{""type"": ""email"", ""value"": ""develo.insercycle...","[{""link"": ""mailto:develo.insercycles@yahoo.com""}]"
3,33771218,botanique-a-la-dune-du-pilat-petites-fleurs-et...,https://openagenda.com/cbnsa/events/botanique-...,Botanique à la dune du Pilat : petites fleurs ...,Sortie botanique de 2h à la Dune du Pilat (Gir...,<p>La célèbre dune cache sous son air désertiq...,Gratuit - inscription obligatoire,None,https://cdn.openagenda.com/main/7838611264464c...,CBNSA,...,46716941,None,None,None,None,None,None,France (Métropole),"[{""type"": ""email"", ""value"": ""c.canton@ladunedu...","[{""link"": ""mailto:c.canton@ladunedupilat.com""}]"
4,40564411,descubrimiento-de-un-jardin-estuche-de-un-tall...,https://openagenda.com/rdvj-2025-nouvelle-aqui...,"Découverte d'un jardin, écrin d'un atelier de ...",Visite du jardin et de l'atelier du sculpteur ...,<p>Visite du jardin et de l'atelier du sculpte...,"Gratuit. Sur réservation, par groupe de 10 max...",None,https://cdn.openagenda.com/main/7541d6c1c0f046...,© François Didier,...,3356551,None,None,None,None,None,None,France (Métropole),"[{""type"": ""phone"", ""value"": ""06 44 71 24 68""},...",NaN


## Colonnes disponibles

On liste les champs retournés pour décider lesquels seront utiles dans le RAG.

In [10]:
list(df.columns)

['uid',
 'slug',
 'canonicalurl',
 'title_fr',
 'description_fr',
 'longdescription_fr',
 'conditions_fr',
 'keywords_fr',
 'image',
 'imagecredits',
 'thumbnail',
 'originalimage',
 'updatedat',
 'daterange_fr',
 'firstdate_begin',
 'firstdate_end',
 'lastdate_begin',
 'lastdate_end',
 'timings',
 'accessibility',
 'accessibility_label_fr',
 'location_uid',
 'location_coordinates',
 'location_name',
 'location_address',
 'location_district',
 'location_insee',
 'location_postalcode',
 'location_city',
 'location_department',
 'location_region',
 'location_countrycode',
 'location_image',
 'location_imagecredits',
 'location_phone',
 'location_website',
 'location_links',
 'location_tags',
 'location_description_fr',
 'location_access_fr',
 'attendancemode',
 'onlineaccesslink',
 'status',
 'age_min',
 'age_max',
 'originagenda_title',
 'originagenda_uid',
 'contributor_email',
 'contributor_contactnumber',
 'contributor_contactname',
 'contributor_contactposition',
 'contributor_organ

## Champs utiles pour le RAG

On garde uniquement les champs qui peuvent aider à recommander ou expliquer un événement : titre, description, dates, lieu, conditions, lien officiel, coordonnées.

In [11]:
available_useful_columns = [col for col in OPENAGENDA_USEFUL_FIELDS if col in df.columns]
missing_useful_columns = [col for col in OPENAGENDA_USEFUL_FIELDS if col not in df.columns]

print("Champs utiles disponibles :", len(available_useful_columns))
print("Champs utiles absents dans cet échantillon :", len(missing_useful_columns))

available_useful_columns

Champs utiles disponibles : 28
Champs utiles absents dans cet échantillon : 0


['uid',
 'slug',
 'canonicalurl',
 'title_fr',
 'description_fr',
 'longdescription_fr',
 'conditions_fr',
 'keywords_fr',
 'daterange_fr',
 'firstdate_begin',
 'firstdate_end',
 'lastdate_begin',
 'lastdate_end',
 'timings',
 'location_name',
 'location_address',
 'location_postalcode',
 'location_city',
 'location_department',
 'location_region',
 'location_countrycode',
 'location_coordinates',
 'accessibility_label_fr',
 'age_min',
 'age_max',
 'registration',
 'onlineaccesslink',
 'originagenda_title']

In [12]:
df_useful = df[available_useful_columns].copy()
df_useful.head()

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,daterange_fr,firstdate_begin,...,location_department,location_region,location_countrycode,location_coordinates,accessibility_label_fr,age_min,age_max,registration,onlineaccesslink,originagenda_title
0,78749329,ballade-velo,https://openagenda.com/maiavelo/events/ballade...,BALLADE VELO -Ô PIGNON,GARE DE CASSY/ ARES BALADE/ REJOINDRE LA FET...,<p>rejoindre fete du velo o pignon ares</p>,NaN,None,"Samedi 10 mai 2025, 06h30",2025-05-10T04:30:00+00:00,...,Gironde,Nouvelle-Aquitaine,FR,"{'lon': -1.054735, 'lat': 44.710912}",None,NaN,NaN,NaN,None,Mai à vélo
1,1832998,lichens-tresors-meconnus,https://openagenda.com/cbnsa/events/lichens-tr...,"Lichens, trésors méconnus",Balade botanique de 2h30 à la découverte des l...,<p>Saviez-vous que certains lichens sont utili...,"Gratuit, inscription obligatoire",None,"Samedi 17 mai 2025, 10h00",2025-05-17T08:00:00+00:00,...,Gironde,Nouvelle-Aquitaine,FR,"{'lon': -1.021883, 'lat': 44.690977}",None,12.0,99.0,"[{""type"": ""link"", ""value"": ""https://www.girond...",None,Conservatoire botanique national Sud-Atlantique
2,15770224,mai-a-velo-sud-bassin-darcachon,https://openagenda.com/maiavelo/events/mai-a-v...,Mai à vélo - Sud Bassin d'Arcachon,Insercycles vous invite à découvrir ou redécou...,<p>Dans le cadre de l'événement national Mai à...,NaN,None,"Samedi 17 mai 2025, 10h00",2025-05-17T08:00:00+00:00,...,Gironde,Nouvelle-Aquitaine,FR,"{'lon': -1.019403, 'lat': 44.639769}",None,NaN,NaN,"[{""type"": ""email"", ""value"": ""develo.insercycle...",None,Mai à vélo
3,33771218,botanique-a-la-dune-du-pilat-petites-fleurs-et...,https://openagenda.com/cbnsa/events/botanique-...,Botanique à la dune du Pilat : petites fleurs ...,Sortie botanique de 2h à la Dune du Pilat (Gir...,<p>La célèbre dune cache sous son air désertiq...,Gratuit - inscription obligatoire,None,"Jeudi 22 mai 2025, 14h00",2025-05-22T12:00:00+00:00,...,Gironde,Nouvelle-Aquitaine,FR,"{'lon': -1.214205, 'lat': 44.588978}",None,NaN,NaN,"[{""type"": ""email"", ""value"": ""c.canton@ladunedu...",None,Conservatoire botanique national Sud-Atlantique
4,40564411,descubrimiento-de-un-jardin-estuche-de-un-tall...,https://openagenda.com/rdvj-2025-nouvelle-aqui...,"Découverte d'un jardin, écrin d'un atelier de ...",Visite du jardin et de l'atelier du sculpteur ...,<p>Visite du jardin et de l'atelier du sculpte...,"Gratuit. Sur réservation, par groupe de 10 max...",None,6 - 8 juin,2025-06-06T07:00:00+00:00,...,Gironde,Nouvelle-Aquitaine,FR,"{'lon': -0.891831, 'lat': 44.484837}",None,NaN,NaN,"[{""type"": ""phone"", ""value"": ""06 44 71 24 68""},...",None,Rendez-vous aux jardins 2025 : Nouvelle-Aquitaine


## Vérification du filtre temporel

On vérifie que les dates retournées sont bien dans la période attendue.

In [13]:
if not df_useful.empty:
    df_useful["firstdate_begin"] = pd.to_datetime(df_useful["firstdate_begin"], errors="coerce")
    print("Date minimale :", df_useful["firstdate_begin"].min())
    print("Date maximale :", df_useful["firstdate_begin"].max())
else:
    print("Aucun événement à vérifier.")

Date minimale : 2025-05-10 04:30:00+00:00
Date maximale : 2025-07-30 17:00:00+00:00


## Vérification du filtre géographique

On vérifie que les villes récupérées correspondent bien à la zone choisie.

In [14]:
if "location_city" in df_useful.columns:
    df_useful["location_city"].value_counts(dropna=False)
else:
    print("La colonne location_city n'est pas présente.")

## Pagination

L'API utilise `limit` pour le nombre de résultats et `offset` pour passer aux pages suivantes.

Cette fonction permet de récupérer plusieurs pages sans encore créer le client final.

In [15]:
def fetch_openagenda_page(limit=OPENAGENDA_PAGE_SIZE, offset=0):
    params = {
        "limit": limit,
        "offset": offset,
        "where": where_clause,
        "order_by": OPENAGENDA_ORDER_BY,
    }

    response = requests.get(OPENAGENDA_BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


page_1 = fetch_openagenda_page(limit=5, offset=0)
page_2 = fetch_openagenda_page(limit=5, offset=5)

print("Page 1 :", len(page_1.get("results", [])), "événements")
print("Page 2 :", len(page_2.get("results", [])), "événements")

Page 1 : 5 événements
Page 2 : 5 événements


## Sauvegarde d'un échantillon brut

In [16]:
sample_path = PATHS.data_raw / RAW_OPENAGENDA_SAMPLE_FILENAME
saved_path = save_openagenda_sample(data, sample_path)

print(f"Échantillon sauvegardé dans : data/raw/{saved_path.name}")

Échantillon sauvegardé dans : data/raw/sample_openagenda.json


## Synthèse des décisions

- Endpoint retenu : `/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records`.
- Aucune clé API nécessaire pour ce POC.
- Utilisation de `where` plutôt que `refine` pour les filtres métier.
- Filtre temporel retenu : `firstdate_begin >= now(years=-1)`.
- Filtre géographique : département Gironde + villes ciblées.
- Champs utiles : titre, descriptions, conditions, dates, lieu, adresse, ville, coordonnées, URL.
- Problèmes observés : descriptions HTML, champs parfois manquants, certains champs JSON stockés sous forme de chaînes.
- La suite consistera à nettoyer les textes et construire un document textuel indexable pour le RAG.